# Chapter 02-02 · Where data comes from

**Label:** Core  |  **Time:** ~45 minutes  |  **Difficulty:** gentle to read, and the habit takes a career

**Prerequisites:** 02-01. You should be able to state what one row represents and classify columns
as target, feature, identifier or metadata.

**Position in the learning path:** module 02, chapter 2 of 8. Before: **02-01**. After: **02-03**,
on the rows you never see.

---

## Why this matters

A dataset is not a window onto the world. It is **a record of a process**: somebody or something
decided what to measure, when, by what method, and for what purpose - and that purpose was almost
certainly not yours.

Every strange thing you will meet in the next six chapters - the sentinel value, the category that
means two things, the gap on Sundays, the impossible zero - has a reason, and the reason lives in
the collection process rather than in the file. You cannot find it by staring at the numbers.

The failure lab is the most expensive version of this: **a change in how something is measured looks
exactly like a change in the world.** A 27% jump in March that any dashboard would celebrate, and
that is entirely an artefact of a firmware update.

## What you will be able to do

By the end of this chapter you can:

1. **Ask** the six provenance questions of any dataset, and say why each one changes your analysis.
2. **Recognise** that a step change in a series is a measurement hypothesis before it is a world
   hypothesis.
3. **Design** a diagnostic that distinguishes the two - a slice of the data where the real quantity
   cannot have changed.
4. **Write** a data dictionary that records what the columns mean and where they came from.
5. **Explain** why data collected for one purpose carries the fingerprints of that purpose.

## Warm-up: retrieve, do not reread

From memory:

1. What is the unit of observation, and how do you choose it?
2. Name the four kinds of column.
3. State the mean-of-means rule.
4. Why is a postcode not a number?

<br>

*Answers: (1) what one row represents; choose it to match the thing you will make a decision about.
(2) target, feature, identifier, metadata. (3) the average of averages is not the average unless
every group is the same size. (4) it is nominal - its digits are names, and support no arithmetic.*

## The situation

Maria's daily rental count comes from a sensor in each docking point. It logs an event when a bike
is **undocked**.

Read that sentence again, because it is doing a lot of work:

- A member of staff moving a bike to a different rack **is an undocking**. Is that a rental?
- A customer who undocks a bike, changes their mind and re-docks it after ten seconds - two events,
  or none?
- A rental that starts at 23:50 and ends at 00:10 belongs to which day?
- A dead battery in one dock means that dock records nothing. Is that zero rentals, or no data?

Nobody wrote these answers down, because the sensor was installed by the dock manufacturer to
support **maintenance scheduling**, not demand analysis. Every definition in the data was chosen for
a purpose that is not yours.

**The question this chapter answers:** what do you have to know about how a number came to exist,
before you are entitled to draw a conclusion from it?

## The six provenance questions

Ask these of every dataset, and write the answers down. They take an afternoon and they are the
highest-return afternoon in any project.

| # | Question | Why it changes your analysis |
|---|---|---|
| 1 | **Who or what created this record, and by what mechanism?** | A sensor, a form, a human, a scheduled job, another model - each fails differently |
| 2 | **Why was it collected?** | Data collected for billing is complete where money changes hands and sloppy elsewhere. Data collected for research is careful about definitions and small |
| 3 | **What exactly does each field mean, in the collector's words?** | "Active customer" means five different things in five departments |
| 4 | **When was it recorded, relative to the event?** | Recorded after the outcome is known is how the future leaks in |
| 5 | **What has changed over time?** | Systems, definitions, thresholds and vendors change. The data does not announce it |
| 6 | **Who or what is missing entirely?** | The rows that were never created are invisible in the file - which is 02-03 |

Question 5 is the one this chapter is built around, because it is the one that produces a *plausible
and wrong* answer rather than an obviously broken one.

**A useful shortcut for question 2:** ask *who would notice if this field were wrong?* A field that
feeds an invoice is watched by a customer and an accountant, and is usually reliable. A field that
feeds nothing is typed once by somebody in a hurry and never read again - and it will be the field
your model finds most predictive, because it encodes who was in a hurry.

## The data

120 days of Maria's recorded rentals. **SYNTHETIC** - we know what really happened, which is the
only way to check whether a diagnostic works.

Two columns: the daily total, and the number of undockings between 2am and 5am, when the stand is
closed and essentially nobody rents a bike.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

rng = np.random.default_rng(12)
n_days = 120
day = pd.date_range("2024-01-01", periods=n_days, freq="D")

true_demand = rng.poisson(100, n_days)          # what customers actually did
true_night = rng.poisson(2, n_days)             # a handful of genuine night rentals

# On 1 March a firmware update started logging staff bike movements as undockings.
upgraded = np.arange(n_days) >= 60
staff_moves = np.where(upgraded, rng.poisson(25, n_days), 0)
staff_at_night = np.where(upgraded, rng.poisson(20, n_days), 0)

recorded = pd.DataFrame({
    "day": day,
    "rentals": true_demand + staff_moves,
    "night_rentals": true_night + staff_at_night,
}).set_index("day")

print(f"firmware update deployed: {day[60].date()}")
recorded.head(3)

### Predict before running

You are the analyst. You do **not** know about the firmware update - nobody told you, because the
person who deployed it works for the dock vendor.

1. What will the monthly average rentals look like?
2. What conclusion will a dashboard draw?
3. What single check could distinguish "demand grew" from "the counter changed"?

In [ ]:
monthly = recorded["rentals"].resample("MS").mean().round(1)
print("mean rentals per day, by month:")
print(monthly.to_string())

before = recorded.loc[:"2024-02-29", "rentals"].mean()
after = recorded.loc["2024-03-01":, "rentals"].mean()
print(f"\nJan-Feb mean {before:.1f}  ->  Mar-Apr mean {after:.1f}   ({after / before - 1:+.0%})")

## Failure lab: the March surge

**Rentals up 27%.** In any organisation this is a good news story: a slide, a congratulation, a
plan to buy more bikes, and a demand forecast retrained on the new higher level.

Every number is correctly computed. The sensor is working exactly as designed. And demand did not
change at all.

Here is the check that finds it.

In [ ]:
night_before = recorded.loc[:"2024-02-29", "night_rentals"].mean()
night_after = recorded.loc["2024-03-01":, "night_rentals"].mean()

print(f"undockings between 2am and 5am:  {night_before:.1f} -> {night_after:.1f} per night "
      f"({night_after / night_before - 1:+.0%})")
print(f"night share of all undockings :  {night_before / before:.1%} -> {night_after / after:.1%}")

In [ ]:
fig, (totals, nights) = plt.subplots(2, 1, figsize=(9, 5.5), sharex=True)

totals.plot(recorded.index, recorded["rentals"], color="#0072B2", linewidth=0.9)
totals.axvline(day[60], color="#D55E00", linestyle="--", label="1 March: firmware update")
totals.set_ylabel("Undockings per day")
totals.set_title("Recorded rentals: a plausible 27% growth story")
totals.legend(fontsize=8)

nights.plot(recorded.index, recorded["night_rentals"], color="#009E73", linewidth=0.9)
nights.axvline(day[60], color="#D55E00", linestyle="--")
nights.set_ylabel("Undockings 2am-5am")
nights.set_xlabel("Date")
nights.set_title("The same days, 2am to 5am: nobody rents a bike at 3am")

fig.tight_layout()
plt.show()

### Diagnosis

**Night undockings went from 1.8 to 20.9 - up more than tenfold.** The night share of all
undockings went from 1.9% to 16.5%.

Real demand cannot do that. Nobody starts renting bicycles at 3am in numbers ten times greater than
before, on a single day, and stops growing immediately afterwards. The only explanations are that
the world changed in a way nobody noticed, or that **the thing being counted changed**.

The night hours are what makes this diagnosable, and the reason is worth stating as a principle:

> **Find a slice of the data where the real quantity cannot plausibly have changed. If it changed
> anyway, the measurement changed.**

Between 2am and 5am the stand is closed, so genuine rentals are near zero and the slice is almost
pure measurement. Staff move bikes at night precisely because the stand is closed - so the
contamination concentrates exactly where the real signal is weakest, which is what makes it visible.

**Three further tells, none of which needs the vendor's help:**

1. **The change is a step, not a ramp.** Real demand growth is gradual and seasonal. A discontinuity
   on one date is a deployment, a policy change, a supplier switch or a definition change.
2. **The date is suspiciously round.** The first of a month, the start of a quarter, the day after a
   release. Real behaviour does not respect the calendar this precisely.
3. **The composition changed, not just the level.** The night *share* tripled. When a total grows for
   real, its parts usually grow roughly together; when a new source is added, one part explodes.

### Remedies

| Remedy | What it catches | Cost |
|---|---|---|
| Keep a **changelog** of the systems that produce your data | Every version of this problem | An hour a quarter, and it must be someone's job |
| Plot every important series over its full history before modelling | Steps, gaps, level shifts, saturation | Two minutes per series |
| Test a slice where the quantity cannot have changed | This exact case | One line, if you can think of the slice |
| Compare *composition*, not just totals | New sources added to an aggregate | One `groupby` |
| Ask the person who runs the system | All of it, faster than any analysis | A conversation |

The last row is the honest one. Every diagnostic in this chapter is a substitute for a question
somebody could have answered in thirty seconds. Analysts systematically under-use it, because asking
feels slower than computing - and it is not.

### What the model would have done

Suppose nobody notices, and a demand forecast is retrained on all 120 days. Two things follow:

- **It learns a level shift that will never repeat**, so its forecast for May is 25 too high in
  expectation - not because the model is bad, but because it was told the world jumped.
- **Any feature correlated with the date after 1 March** - a marketing campaign, a price change, a
  new station - will be credited with the jump. The model will confidently attribute a firmware
  update to whatever else happened that month, and someone will spend money repeating it. That is
  00-04's confounding, arriving through the collection process.

---

## The data dictionary

The output of provenance work is a document, not a feeling. A **data dictionary** records what each
column means and where it came from, so that the next person - who is usually you, in four months -
does not have to rediscover it.

The minimum useful entry for a column:

| Field | Example |
|---|---|
| Name | `rentals` |
| Meaning, in the collector's words | "count of undocking events at any dock at this station on this date" |
| Units | events (a count, not a customer count) |
| Source system | Dock firmware v2.x, exported nightly by the vendor |
| Time semantics | Date the undocking occurred, in local time; a rental across midnight counts on its start date |
| Known issues | Since 1 March 2024, includes staff movements. Not comparable across that date |
| Missing-value convention | A dock with a dead battery produces no row - absent, not zero |

That "known issues" row is the point of the whole exercise. It is the sentence that stops the next
person repeating your March surprise.

In [ ]:
dictionary = pd.DataFrame([
    {"column": "rentals", "meaning": "undocking events per station-day", "units": "events",
     "known_issue": "includes staff moves from 2024-03-01; not comparable across that date"},
    {"column": "night_rentals", "meaning": "undockings between 02:00 and 05:00", "units": "events",
     "known_issue": "useful as a contamination check; near zero when only customers are counted"},
    {"column": "day", "meaning": "local calendar date of the undocking", "units": "date",
     "known_issue": "rentals crossing midnight are attributed to the start date"},
])
print(dictionary.to_string(index=False))

Three columns, three sentences each, and it took a minute. It is also the single most useful artefact
you can leave behind on a project - more useful than the model, which will be retrained, and more
useful than the notebook, which nobody will read.

Chapter 13-03 turns this into a **data card**, the published version. This is the working version,
and it should exist from day one.

## Purpose leaves fingerprints

Question 2 - *why was it collected?* - deserves its own section, because the answer predicts what
will be wrong before you look.

| Collected for | Reliable where | Unreliable where |
|---|---|---|
| **Billing** | Anything a customer disputes or an auditor checks | Anything free, cancelled, or written off |
| **Operations** | What the operators need to do their job today | History, because old rows get overwritten or purged |
| **Compliance** | The specific fields a regulator inspects | Everything else, filled in to satisfy a required field |
| **Marketing** | Whoever engages | Whoever ignores you - and the two groups differ systematically |
| **Research** | Definitions, because someone wrote a protocol | Coverage - it is usually small and specific |
| **A previous model** | Nothing, by default | Everything: the values reflect the old model's decisions, not the world |

The last row is the one that catches experienced people. If a table records "risk score" produced by
last year's model, training a new model on it teaches the new model to imitate the old one,
including its mistakes. And if the old model's decisions changed who got a loan, or who was
inspected, or who was called - then the *outcomes* in your data are consequences of those decisions.
That is a feedback loop, and it is 11-07 and 13-08.

**A concrete instance you will meet:** a fraud dataset labelled "confirmed fraud" contains only the
fraud that was *caught*, by a system with its own blind spots. A model trained on it learns to find
the fraud that the old system already finds.

## Common misconceptions

**"Provenance is documentation - a nice-to-have."**
It is a modelling input. Whether the March jump is real determines whether you retrain, what your
forecast says, and what the business spends. Nothing in the file answers it.

**"The data is what it is; I'll let the model figure it out."**
A model cannot distinguish a firmware update from a demand surge. It has no access to the world,
only to the file, and in the file the two are identical.

**"A step change means something happened."**
Something happened *to the data*. Whether it happened to the world is a separate question with a
different kind of evidence.

**"If it's in the database, someone validated it."**
Someone validated the fields their job depends on. Everything else is a required box that had to be
filled in.

**"Old data and new data are the same data."**
Definitions drift. "Active user" is redefined, a threshold moves, a vendor changes, a field is
repurposed because adding a column needed a migration. Long histories are the *most* likely to
contain silent definition changes, not the least.

**"I can reconstruct the meaning from the values."**
Sometimes you can guess. But the difference between "no rentals" and "no data" is invisible in a
column of zeros, and guessing wrong invents observations. Ask.

---

## Exercises

Solutions: `solutions/02_data_literacy/02-02_provenance_solutions.ipynb`.

### Quick understanding

**E1 (define).** List the six provenance questions from memory, and say which one the March surge
turned on.

**E2 (explain).** Why is a slice where "the real quantity cannot have changed" such a powerful
diagnostic? Give one example from a domain other than bike rentals.

**E3 (explain).** Why is data collected for billing reliable in some places and not others? Give one
field of each kind for an online shop.

### Hand calculation

**E4 (calculate).** Before the update: 99 rentals per day, of which 1.8 at night. After: 126 per
day, of which 20.9. Compute the night share before and after, and the implied number of staff moves
per day if all of the night increase is staff. Then estimate what fraction of the 27% total increase
the staff moves explain.

**E5 (calculate).** A dashboard reports "March was our best month, up 27% on February". Rewrite that
sentence three ways: as it would be if the growth were real, as it should be written given what we
know, and as it would be written by someone who wanted the bonus.

### Coding

**E6 (code).** Write `find_step(series, min_gap=0.15)` that scans every possible split point,
compares the mean before and after, and returns the date with the largest relative jump. Run it on
`rentals` and on `night_rentals`. Do they agree?

**E7 (code).** Build the corrected series: estimate the daily staff contamination from the night
counts and subtract it from the totals. Plot the corrected series against the original, and say what
assumption your correction rests on.

### Interpretation

**E8 (interpret).** Your correction in E7 relies on an assumption that could be wrong. Name it,
describe a situation in which it fails, and say what evidence would let you check it.

### Debugging

**E9 (diagnose).** A churn model's accuracy drops sharply for customers who joined after a certain
month. Nobody changed the model. Give four candidate provenance explanations, ordered by how cheap
they are to check, and say what you would ask first.

### Exam and interview reasoning

**E10 (defend).** *"Why do you spend the first day of a project talking to people instead of
modelling?"* Answer in about 120 words with a concrete example.

**E11 (design).** You are handed five years of hospital admissions to build a readmission model.
List the five provenance questions you would ask first, say who you would ask each one, and name the
answer that would most change your approach.

### Transfer to a different situation

**E12 (design).** A retailer's "customer satisfaction score" jumped from 3.8 to 4.4 in one week. Give
three measurement explanations before any explanation about customers, and describe the slice you
would check for each.

### Explain it to someone non-technical

**E13 (explain).** In under 80 words, explain to Maria why you do not believe the 27% and what you
want to check. Do not use the words "data", "metric" or "artefact".

### Optional challenge

**E14 (code + diagnose).** Simulate the cost of missing this. Fit a simple trend model on all 120
days and forecast the next 30, then fit the same model on the corrected series. Compare the two
forecasts against the true demand, and express the difference in bikes - the number Maria would
over-order.

In [ ]:
# Your workspace. Still in memory: recorded, day, true_demand, true_night, staff_moves, upgraded.

## Mastery check

Without scrolling up, can you:

- [ ] Recite at least four of the six provenance questions? *(If not: "The six provenance
      questions".)*
- [ ] Say why a step change is a measurement hypothesis first? *(If not: "Failure lab".)*
- [ ] Describe the "slice that cannot have changed" diagnostic? *(If not: "Diagnosis".)*
- [ ] Name the seven fields of a data dictionary entry? *(If not: "The data dictionary".)*
- [ ] Say why data from a previous model is the most dangerous source? *(If not: "Purpose leaves
      fingerprints".)*

## What should now feel instinctive

1. **"How did this number come to exist?"** - asked before "what does this number say?".
2. **A step in a series is a deployment until proven otherwise.** Check the date against a
   changelog before checking it against a business story.
3. **Look for a slice where the truth cannot have moved.** Night hours, closed days, a control
   region, a segment the change should not touch.
4. **Write the data dictionary as you learn it**, not at the end when you have forgotten.
5. **Ask the person who runs the system.** It is faster than every diagnostic in this chapter.

## Flashcards

| Question | Answer |
|---|---|
| What is a dataset? | A record of a collection process, not a window onto the world |
| The six provenance questions | Who/what created it; why; what each field means; when recorded relative to the event; what changed over time; who is missing |
| A step change in a series means? | The measurement changed, until you have evidence that the world did |
| The diagnostic for a measurement change | Find a slice where the real quantity cannot have changed, and see whether it changed |
| Three tells of a definition change | A step rather than a ramp; a suspiciously round date; composition shifting, not just level |
| Why does purpose matter? | Data is careful where someone would notice it being wrong, and careless elsewhere |
| Most dangerous data source | Output of a previous model - it records that model's decisions, not the world |
| What must a data dictionary record? | Meaning in the collector's words, units, source system, time semantics, known issues, missing-value convention |
| Fastest provenance tool | Asking the person who runs the system |
| What would the model have done? | Learned a level shift that never repeats, and credited it to whatever else happened that month |

## Next

**02-03 · Sampling and selection bias: the rows you never see.**

Provenance question 6 was *who is missing entirely?* - and it is the only one you cannot answer by
looking at the file, because the evidence is precisely what is not in it. A dataset of customers who
answered a survey, patients who were tested, or bikes that were returned is a dataset about the
people and things that made it through a filter. The next chapter is about seeing that filter.

New terms are in [GLOSSARY.md](../../GLOSSARY.md).